# Article 6 — Robustesse des modèles de segmentation dentaire aux artefacts physiques de la panoramique

**Avant de lancer :**
1. Menu **Exécution → Modifier le type d'exécution → GPU (T4)**.
2. Dans votre Google Drive, créez le dossier **`MyDrive/Article6`** et copiez-y (depuis `C:\Users\HP\Desktop\panoramique data\Article6`) :
   - le dossier `images/` (les PNG anonymisés),
   - `metadata_157.csv`,
   - `grille_annotation_artefacts-remplis.xlsx`,
   - le dossier `code/` (contient `pano_artifacts.py`).
   **Ne copiez jamais** `CONFIDENTIEL_correspondance.csv`.
3. Exécutez les cellules dans l'ordre (**Exécution → Tout exécuter**).

Le notebook reprend là où il s'est arrêté si la session Colab est coupée : les modèles déjà entraînés sont sauvegardés dans `MyDrive/Article6/resultats/`.

| Expérience | Ce qu'elle mesure | Données |
|---|---|---|
| **E1** | performance de référence | test interne STS-2D-Tooth |
| **E2** | chute de performance par artefact et par sévérité (contrôlée) | test STS + simulateur physique |
| **E3** | comportement sur vos images Owandy réelles, selon les artefacts annotés par le dentiste | 151 images Owandy |

Comparaison principale : **augmentation générique** vs **augmentation physique** (simulateur `pano_artifacts.py`), pour 3 architectures.

In [ ]:
# === 0. Installation et configuration ===
!pip -q install segmentation-models-pytorch datasets openpyxl timm
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time, random, glob
import numpy as np, cv2, pandas as pd, torch
import torch.nn as nn

ROOT = '/content/drive/MyDrive/Article6'
OUT = f'{ROOT}/resultats'
os.makedirs(f'{OUT}/ckpt', exist_ok=True)
sys.path.insert(0, f'{ROOT}/code')
import pano_artifacts as PA

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Appareil :', DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '(PAS DE GPU : activez-le dans Exécution > Type d\'exécution)')

SEED = 42
H, W = 512, 1024          # taille d'entrée (rapport ~1:2 comme une panoramique)
EPOCHS = 40               # réduire à 20 pour un essai rapide
BATCH = 4
LR = 3e-4

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()

In [ ]:
# === 1. Chargement de STS-2D-Tooth (Wang et al., Scientific Data 2025, CC-BY-4.0) ===
from datasets import load_dataset

def load_sts():
    for rid in ['Angelou0516/STS-2D-Tooth', 'MedOtter/STS-2D-Tooth']:
        try:
            a = load_dataset(rid, split='a_pxi_labeled')
            c = load_dataset(rid, split='c_pxi_labeled')
            print('Source :', rid, '| adultes', len(a), '| enfants', len(c))
            return a, c
        except Exception as e:
            print(rid, '->', repr(e)[:200])
    raise RuntimeError('STS-2D-Tooth indisponible : vérifiez la connexion ou le nom du dépôt.')

def to_np(ex):
    im = np.array(ex['image'].convert('L'))
    mk = (np.array(ex['mask'].convert('L')) > 127).astype(np.uint8)
    im = cv2.resize(im, (W, H), interpolation=cv2.INTER_AREA)
    mk = cv2.resize(mk, (W, H), interpolation=cv2.INTER_NEAREST)
    return im, mk

a_ds, c_ds = load_sts()
X, Y, GRP = [], [], []
for ds, g in [(a_ds, 'adulte'), (c_ds, 'enfant')]:
    for ex in ds:
        im, mk = to_np(ex); X.append(im); Y.append(mk); GRP.append(g)
X = np.stack(X); Y = np.stack(Y); GRP = np.array(GRP)
if Y.mean() > 0.5:                      # sécurité : dents = 1
    Y = 1 - Y
print('Images :', X.shape, '| fraction de pixels "dent" :', round(float(Y.mean()), 3))

rs = np.random.RandomState(SEED)
idx = rs.permutation(len(X)); n = len(X)
TE, VA, TR = idx[:int(.1 * n)], idx[int(.1 * n):int(.2 * n)], idx[int(.2 * n):]
print('train / val / test :', len(TR), len(VA), len(TE))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].imshow(X[TR[0]], cmap='gray'); ax[0].set_title('Image STS')
ax[1].imshow(X[TR[0]], cmap='gray'); ax[1].imshow(Y[TR[0]], alpha=.4, cmap='autumn'); ax[1].set_title('Masque dents')
for a_ in ax: a_.axis('off')
plt.show()

In [ ]:
# === 2. Augmentations : générique vs physique ===
def generic_aug(im, mk, rng):
    if rng.random() < .5:
        im, mk = im[:, ::-1].copy(), mk[:, ::-1].copy()
    ang, sc = rng.uniform(-5, 5), rng.uniform(.9, 1.1)
    M = cv2.getRotationMatrix2D((W / 2, H / 2), ang, sc)
    M[:, 2] += (rng.uniform(-.03, .03) * W, rng.uniform(-.03, .03) * H)
    im = cv2.warpAffine(im, M, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    mk = cv2.warpAffine(mk, M, (W, H), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_REFLECT)
    f = im.astype(np.float32) * rng.uniform(.8, 1.2) + rng.uniform(-20, 20)
    if rng.random() < .3:
        f = cv2.GaussianBlur(f, (0, 0), rng.uniform(.5, 1.5))
    if rng.random() < .3:
        f = f + rng.normal(0, rng.uniform(2, 8), f.shape)
    return np.clip(f, 0, 255).astype(np.uint8), mk

def physics_aug(im, mk, rng):
    im, mk = generic_aug(im, mk, rng)
    if rng.random() < .6:
        im, mk, _ = PA.random_augment(im, mk, n=(1, 3), rng=rng)
    return im, mk

AUGS = {'generique': generic_aug, 'physique': physics_aug}

MEAN, STD = 0.45, 0.225
def to_tensor(im):
    x = (im.astype(np.float32) / 255. - MEAN) / STD
    return torch.from_numpy(np.repeat(x[None], 3, 0))

class SegDS(torch.utils.data.Dataset):
    def __init__(self, ids, aug=None):
        self.ids, self.aug = ids, aug
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        im, mk = X[self.ids[i]], Y[self.ids[i]]
        if self.aug is not None:
            rng = np.random.default_rng(np.random.randint(0, 2**31 - 1))
            im, mk = self.aug(im, mk, rng)
        return to_tensor(im), torch.from_numpy(mk[None].astype(np.float32))

def wif(worker_id):
    np.random.seed((torch.initial_seed() + worker_id) % 2**32)

# aperçu de l'augmentation physique
rng = np.random.default_rng(0)
fig, ax = plt.subplots(2, 3, figsize=(18, 6))
for a_ in ax.ravel():
    im, mk = physics_aug(X[TR[1]], Y[TR[1]], rng)
    a_.imshow(im, cmap='gray'); a_.contour(mk, [.5], colors='r', linewidths=.5); a_.axis('off')
plt.suptitle('Exemples d\'augmentation physique (contour rouge = masque déformé avec l\'image)')
plt.show()

In [ ]:
# === 3. Modèles, métriques, entraînement ===
import segmentation_models_pytorch as smp
from scipy import ndimage

MODELS = {
    'unet_resnet50': lambda: smp.Unet('resnet50', encoder_weights='imagenet', in_channels=3, classes=1),
    'unetpp_effb4':  lambda: smp.UnetPlusPlus('efficientnet-b4', encoder_weights='imagenet', in_channels=3, classes=1),
    'fpn_mitb2':     lambda: smp.FPN('mit_b2', encoder_weights='imagenet', in_channels=3, classes=1),  # encodeur Transformer
}

def dice_np(p, g):
    p, g = p.astype(bool), g.astype(bool)
    s = p.sum() + g.sum()
    return 1.0 if s == 0 else 2 * (p & g).sum() / s

def iou_np(p, g):
    p, g = p.astype(bool), g.astype(bool)
    u = (p | g).sum()
    return 1.0 if u == 0 else (p & g).sum() / u

def hd95_np(p, g):
    p, g = p.astype(bool), g.astype(bool)
    if p.sum() == 0 or g.sum() == 0:
        return np.nan
    pb = p ^ ndimage.binary_erosion(p); gb = g ^ ndimage.binary_erosion(g)
    dg = ndimage.distance_transform_edt(~gb); dp = ndimage.distance_transform_edt(~pb)
    return float(np.percentile(np.concatenate([dg[pb], dp[gb]]), 95))

dice_loss = smp.losses.DiceLoss('binary', from_logits=True)
bce = nn.BCEWithLogitsLoss()

@torch.no_grad()
def predict(model, ims, tta=True, bs=4):
    """ims : liste/array d'images uint8 HxW -> probabilités float32 HxW."""
    model.eval(); out = []
    for i in range(0, len(ims), bs):
        x = torch.stack([to_tensor(im) for im in ims[i:i + bs]]).to(DEV)
        with torch.autocast(device_type='cuda', enabled=DEV == 'cuda'):
            p = torch.sigmoid(model(x))
            if tta:
                p = (p + torch.sigmoid(model(x.flip(-1))).flip(-1)) / 2
        out.append(p[:, 0].float().cpu().numpy())
    return np.concatenate(out)

def train_one(mname, aname):
    tag = f'{mname}__{aname}'
    ck = f'{OUT}/ckpt/{tag}.pt'
    done = f'{OUT}/ckpt/{tag}.done.json'
    try:
        model = MODELS[mname]().to(DEV)
    except Exception as e:
        print(f'[{tag}] architecture indisponible avec cette version de smp : {e}')
        return None
    if os.path.exists(done):
        model.load_state_dict(torch.load(ck, map_location=DEV)); print(f'[{tag}] déjà entraîné -> chargé')
        return model
    seed_all()
    dl = torch.utils.data.DataLoader(SegDS(TR, AUGS[aname]), batch_size=BATCH, shuffle=True,
                                     num_workers=2, worker_init_fn=wif, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=DEV == 'cuda')
    best, log, t0 = -1, [], time.time()
    for ep in range(EPOCHS):
        model.train(); tl = 0
        for x, y in dl:
            x, y = x.to(DEV), y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', enabled=DEV == 'cuda'):
                lo = model(x); loss = dice_loss(lo, y) + bce(lo, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item()
        sch.step()
        pv = predict(model, X[VA], tta=False)
        vd = float(np.mean([dice_np(p > .5, g) for p, g in zip(pv, Y[VA])]))
        log.append(dict(epoch=ep + 1, loss=tl / len(dl), val_dice=vd))
        if vd > best:
            best = vd; torch.save(model.state_dict(), ck)
        print(f'[{tag}] époque {ep + 1}/{EPOCHS}  perte {tl / len(dl):.4f}  Dice val {vd:.4f}  ({(time.time() - t0) / 60:.1f} min)')
    pd.DataFrame(log).to_csv(f'{OUT}/ckpt/{tag}_log.csv', index=False)
    json.dump(dict(best_val_dice=best, minutes=(time.time() - t0) / 60, epochs=EPOCHS), open(done, 'w'))
    model.load_state_dict(torch.load(ck, map_location=DEV))
    return model

In [ ]:
# === 4. Entraînement des 6 modèles (3 architectures x 2 augmentations) ===
# Durée indicative sur T4 : 30 à 60 min par modèle. Si la session coupe, relancez cette cellule :
# les modèles terminés sont rechargés, l'entraînement reprend au suivant.
TRAINED = {}
for m in MODELS:
    for a in AUGS:
        mod = train_one(m, a)
        if mod is not None:
            TRAINED[(m, a)] = mod
print('Modèles prêts :', list(TRAINED))

In [ ]:
# === 5. E1 — test interne STS ===
rows = []
for (m, a), mod in TRAINED.items():
    P = predict(mod, X[TE])
    for k, (p, g) in enumerate(zip(P, Y[TE])):
        pb = p > .5
        rows.append(dict(modele=m, augmentation=a, image=int(TE[k]), groupe=GRP[TE[k]],
                         dice=dice_np(pb, g), iou=iou_np(pb, g), hd95_px=hd95_np(pb, g)))
E1 = pd.DataFrame(rows); E1.to_csv(f'{OUT}/E1_test_interne.csv', index=False)
print(E1.groupby(['modele', 'augmentation'])[['dice', 'iou', 'hd95_px']].mean().round(4))

In [ ]:
# === 6. E2 — robustesse contrôlée : chaque artefact, 3 niveaux de sévérité ===
SEV = {
    'menton_bas':        [dict(k=.04), dict(k=.08), dict(k=.12)],
    'menton_haut':       [dict(k=.03, palate=.10), dict(k=.06, palate=.20), dict(k=.10, palate=.35)],
    'inclinaison':       [dict(t=.02, mag=.015), dict(t=.04, mag=.03), dict(t=.06, mag=.05)],
    'rotation':          [dict(r=.07), dict(r=.13), dict(r=.20)],
    'trop_en_avant':     [dict(s=-.12, blur=3), dict(s=-.23, blur=6), dict(s=-.35, blur=9)],
    'trop_en_arriere':   [dict(s=.12, blur=3), dict(s=.23, blur=6), dict(s=.35, blur=9)],
    'fantome_rachis':    [dict(a=.25, width=.08), dict(a=.45, width=.11), dict(a=.65, width=.14)],
    'mouvement':         [dict(x0=.5, shift=.013), dict(x0=.5, shift=.026), dict(x0=.5, shift=.04)],
    'fantome_metallique':[dict(amp=.15, hmag=1.5, up=.04, source='boucle'), dict(amp=.30, hmag=2.0, up=.08, source='boucle'),
                          dict(amp=.45, hmag=2.5, up=.12, source='boucle')],
    'dose_inadaptee':    [dict(gamma=.92), dict(gamma=.81), dict(gamma=.70)],
}
# images dégradées (identiques pour tous les modèles)
CASES = []
for art, levels in SEV.items():
    for lv, prm in enumerate(levels, 1):
        ims, gts = [], []
        for j, t in enumerate(TE):
            rng = np.random.default_rng(1000 * lv + j)
            im, mk, _ = PA.apply(art, X[t], Y[t], rng, **prm)
            ims.append(im); gts.append(mk)
        CASES.append((art, lv, np.stack(ims), np.stack(gts)))
rows = []
for (m, a), mod in TRAINED.items():
    P0 = predict(mod, X[TE])
    for j, t in enumerate(TE):
        rows.append(dict(modele=m, augmentation=a, artefact='aucun', niveau=0, image=int(t),
                         dice=dice_np(P0[j] > .5, Y[t]), hd95_px=hd95_np(P0[j] > .5, Y[t])))
    for art, lv, ims, gts in CASES:
        P = predict(mod, ims)
        for j, t in enumerate(TE):
            rows.append(dict(modele=m, augmentation=a, artefact=art, niveau=lv, image=int(t),
                             dice=dice_np(P[j] > .5, gts[j]), hd95_px=hd95_np(P[j] > .5, gts[j])))
    print('E2 terminé pour', m, a)
E2 = pd.DataFrame(rows); E2.to_csv(f'{OUT}/E2_robustesse_controlee.csv', index=False)
print(E2.pivot_table(index=['artefact', 'niveau'], columns=['modele', 'augmentation'], values='dice').round(3))

In [ ]:
# === 7. E3 — images Owandy réelles ===
meta = pd.read_csv(f'{ROOT}/metadata_157.csv')
meta = meta[meta.duplicate_of.isna() & meta.flag.isna()]
ann = pd.read_excel(f'{ROOT}/grille_annotation_artefacts-remplis.xlsx', sheet_name='Annotation')
ann = ann[ann['Qualité globale (1-3)'].notna()]
df = meta.merge(ann, on=['image_id', 'patient_id'], how='inner')
print('Images Owandy analysées :', len(df))

def load_real(iid):
    a = cv2.imread(f'{ROOT}/images/{iid}.png', 0)
    return cv2.resize(a, (W, H), interpolation=cv2.INTER_AREA), a.shape

REAL = {iid: load_real(iid) for iid in df.image_id}
ims = np.stack([REAL[i][0] for i in df.image_id])
rows = []
for (m, a), mod in TRAINED.items():
    P = predict(mod, ims, tta=False); Pf = predict(mod, ims[:, :, ::-1].copy(), tta=False)[:, :, ::-1]
    for k, iid in enumerate(df.image_id):
        p = (P[k] + Pf[k]) / 2; pb = p > .5
        nlab = ndimage.label(pb)[1]
        unc = float((4 * p * (1 - p))[pb | (p > .2)].mean()) if (p > .2).any() else np.nan
        tta = float(np.abs(P[k] - Pf[k])[pb].mean()) if pb.any() else np.nan
        rows.append(dict(modele=m, augmentation=a, image_id=iid, aire_dents=float(pb.mean()),
                         n_composantes=int(nlab), incertitude=unc, desaccord_tta=tta))
E3 = pd.DataFrame(rows).merge(df, on='image_id')
E3.to_csv(f'{OUT}/E3_owandy_reel.csv', index=False)
print(E3.groupby(['modele', 'augmentation'])[['incertitude', 'desaccord_tta', 'aire_dents']].mean().round(4))

In [ ]:
# === 8. Pré-masques pour correction par le dentiste (50 images) ===
# Meilleur modèle selon E1, prédiction remise à la taille d'origine.
best = E1.groupby(['modele', 'augmentation']).dice.mean().idxmax()
print('Modèle utilisé pour les pré-masques :', best)
rs = np.random.RandomState(SEED)
sel = list(rs.choice(df.image_id.values, size=min(50, len(df)), replace=False))
os.makedirs(f'{OUT}/pre_masques', exist_ok=True); os.makedirs(f'{OUT}/pre_masques_apercu', exist_ok=True)
Psel = predict(TRAINED[best], np.stack([REAL[i][0] for i in sel]))
for iid, p in zip(sel, Psel):
    h0, w0 = REAL[iid][1]
    mk = (cv2.resize(p, (w0, h0), interpolation=cv2.INTER_LINEAR) > .5).astype(np.uint8) * 255
    cv2.imwrite(f'{OUT}/pre_masques/{iid}_masque.png', mk)
    im0 = cv2.imread(f'{ROOT}/images/{iid}.png', 0)
    ov = cv2.cvtColor(im0, cv2.COLOR_GRAY2BGR)
    cnt, _ = cv2.findContours(mk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(ov, cnt, -1, (0, 0, 255), 2)
    cv2.imwrite(f'{OUT}/pre_masques_apercu/{iid}_apercu.jpg', cv2.resize(ov, None, fx=.5, fy=.5))
pd.Series(sel, name='image_id').to_csv(f'{OUT}/pre_masques/liste_50_images.csv', index=False)
print('Pré-masques écrits dans', f'{OUT}/pre_masques')

In [ ]:
# === 9. Archive des résultats à me renvoyer ===
import shutil
PACK = '/content/pack'; shutil.rmtree(PACK, ignore_errors=True); os.makedirs(f'{PACK}/logs', exist_ok=True)
for f in glob.glob(f'{OUT}/E*.csv'):
    shutil.copy(f, PACK)
for f in glob.glob(f'{OUT}/ckpt/*_log.csv') + glob.glob(f'{OUT}/ckpt/*.done.json'):
    shutil.copy(f, f'{PACK}/logs')
shutil.make_archive(f'{ROOT}/resultats_article6', 'zip', PACK)    # sans les poids des modèles (lourds)
print('Archive prête :', f'{ROOT}/resultats_article6.zip')
print('Copiez ce fichier zip dans C:\\Users\\HP\\Desktop\\panoramique data\\Article6 et prévenez Claude.')
print('Les pré-masques (dossier resultats/pre_masques) sont pour le dentiste.')